In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyro
import pyro.distributions as dist
import scipy.io as sio
import torch
import torch.nn as nn
from pyro.infer import MCMC, NUTS, Predictive
from pyro.infer.autoguide import AutoLowRankMultivariateNormal
from pyro.nn import PyroModule, PyroSample
from pyro.optim import Adam

from src.dataset import ESNDataset, construct_dataset, partition_dataset
from src.methods import ESNMCMC, ESNQR, ESNVariational
from src.models import MLP, SSVS, BayesianModel, BayesianNeuralNetwork
from src.reservoir import Reservoir

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Synthetic

# Real

In [3]:
# Load the MAT file for training and validation
rest1_file = sio.loadmat("./data/100307.REST1.LR.SchaeferS.ptseries.mat")
rest2_file = sio.loadmat("./data/100307.REST2.LR.SchaeferS.ptseries.mat")
data_rest1 = torch.from_numpy(rest1_file["tseries"]).float().t()
data_rest2 = torch.from_numpy(rest2_file["tseries"]).float().t()

In [4]:
mu1 = torch.mean(data_rest1, dim=0)
std1 = torch.std(data_rest1, dim=0)
mu2 = torch.mean(data_rest2, dim=0)
std2 = torch.std(data_rest2, dim=0)

data_rest1 = (data_rest1 - mu1) / std1
data_rest2 = (data_rest2 - mu2) / std2

## Create the Reservoir

In [5]:
num_neurons_real = 30
num_features_real = data_rest1.shape[1]
reservoir_real = Reservoir(
    input_features=num_features_real, num_neurons=num_neurons_real
)

## Generate the $(s,x)$ dataset

In [6]:
burnin_steps = 50
states_rest1 = torch.zeros((data_rest1.shape[0] - 1, num_neurons_real))
states_rest2 = torch.zeros((data_rest2.shape[0] - 1, num_neurons_real))

for i, x in enumerate(data_rest1[:-1]):
    states_rest1[i, :] = reservoir_real(x)

reservoir_real.reset_state()

for i, x in enumerate(data_rest2[:-1]):
    states_rest2[i, :] = reservoir_real(x)

dataset_rest1 = ESNDataset(
    states_rest1[burnin_steps:], data_rest1[burnin_steps:], mu1, std1
)
dataset_rest2 = ESNDataset(
    states_rest2[burnin_steps:], data_rest2[burnin_steps:], mu2, std2
)

In [7]:
train_set, cal_set, test_set = partition_dataset([dataset_rest1, dataset_rest2])

IndexError: index 1149 is out of bounds for dimension 0 with size 1149

## Create the basic model

In [ ]:
bayes_model_real = BayesianModel(
    in_features=num_neurons_real, out_features=num_features_real
)

## MCMC

In [ ]:
esn_mcmc = ESNMCMC(bayes_model_real)

In [ ]:
# mcmc hyperparameters
warmup_steps = 150
num_samples = 20
num_chains = 5
mc_context = "spawn"

esn_mcmc.run(
    train_set,
    warmup_steps=warmup_steps,
    num_samples=num_samples,
    num_chains=num_chains,
    mc_context=mc_context,
)

Warmup [1]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [2]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [3]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [4]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [5]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [6]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [7]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [8]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [9]:   0%|          | 0/160 [00:00, ?it/s]

Warmup [10]:   0%|          | 0/160 [00:00, ?it/s]

In [ ]:
y_pred = esn_mcmc.predict(test_set)

## SVI

In [ ]:
# Setup Optimizer
LR = 0.001
EPOCHS = 100

optimizer = pyro.optim.Adam({"lr": LR})

In [ ]:
# PYRO GUIDE

RANK = int(np.sqrt(num_neurons_real))  # Rank for the Multivariate Normal
guide = AutoLowRankMultivariateNormal(bayes_model_real, rank=RANK)

In [ ]:
esn_var_real = ESNVariational(
    bayes_model_real,
    guide,
    optimizer,
)

## QR

In [ ]:
class CheckLoss(nn.Module):
    def __init__(self, taus: list[float]) -> None:
        super().__init__()
        self.taus = taus
        return

    def forward(self, predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss = torch.zeros(1)
        # Loop over each tau
        for i, tau in enumerate(self.taus):
            # Compute the residuals
            r = targets - predictions[:, i].unsqueeze(1)

            # Compute the loss for a given tau
            quantile_loss = tau * torch.relu(r) + (1 - tau) * torch.relu(-r)
            loss += torch.mean(quantile_loss)
        return loss

In [ ]:
layer_dims = [num_neurons_real, num_features_real]
mlp = MLP(layer_dims, nn.ReLU())

In [ ]:
lr = 0.01
taus = [0.05, 0.5, 0.95]
esn_qr = ESNQR(mlp, torch.optim.Adam(mlp.parameters(), lr), CheckLoss(taus))

## SVSS

In [ ]:
ssvs_model_real = SSVS(num_features_real)

In [ ]:
esn_mcmc = ESNMCMC(ssvs_model_real)